In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [ ]:
# python standard library imports
from typing import Self, Any
from pathlib import Path
import json
import random
import tensorflow as tf
from keras import Model, Sequential, Input, layers
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler
from keras.utils import image_dataset_from_directory
import matplotlib.pyplot as plt

In [3]:
seed = 123
random.seed(seed)
tf.random.set_seed(seed)

In [4]:
# Load the class weights and convert keys back to integers
with open("class_weights.json", "r") as f:
    loaded_weights = {int(k): v for k, v in json.load(f).items()}

In [5]:
class MyCNN(Model):
    """
    MyCNN class, inherets from keras' Model class
    """
    def __init__(self: Self, augmentation_layer, conv_configs, dense_configs, num_classes, activation: str = "relu"):
        """
        Initialization
        """

        super().__init__(name="my_cnn")
        
        self.augmentation_layer = augmentation_layer

        self.blocks = []
        for id, (filters, kernel, stride) in enumerate(conv_configs):
            block = {
                'conv': layers.Conv2D(filters, kernel, strides=stride, name=f"conv_layer_{id}", padding='same'),
                'bn': layers.BatchNormalization(),
                'actv': layers.Activation(activation),
                # Projection 1x1 to align channels/dimensions if needed
                'shortcut': layers.Conv2D(filters, (1, 1), strides=stride, padding='same')
            }
            self.blocks.append(block)

        # Global Average Pooling instead of flattening to reduce spatial dimensions and number of parameters
        self.gap = layers.GlobalAveragePooling2D(name="GAP_layer")
        
        self.dense_layers = [layers.Dense(u, name=f"dense_layer_{id}", activation=activation) for id, u in enumerate(dense_configs)]
        self.classifier = layers.Dense(num_classes, name="classification_head", activation='softmax')

    def call(self, inputs, training=False):
        """
        Forward call
        """
        # 1. Apply augmentation
        x = self.augmentation_layer(inputs, training=training)

        # 2. Pass through convolutional blocks with residual connections
        for b in self.blocks:
            shortcut = b['shortcut'](x)

            # We use the augmented 'x' here, so the convolutional block learns to refine the augmented features
            x_conv = b['conv'](x)
            x_bn = b['bn'](x_conv, training=training)
            x_actv = b['actv'](x_bn)

            # Residual connection: add the shortcut to the activated output of the block
            x_add = layers.Add()([x_actv, shortcut])
            x = b['actv'](x_add) # Activation after addition

        # 3. Global Average Pooling to reduce spatial dimensions
        x = self.gap(x)
        for layer in self.dense_layers:
            x = layer(x)
            
        return self.classifier(x)

In [6]:
seed = 123
random.seed(seed)
tf.random.set_seed(seed)

In [7]:
# Hyperparameters and dataset loading
image_size = (224, 224) # Can be adjusted based on model capacity and dataset
epochs = 64
batch_size = 32
data_dir_path = "wikiart_split"

train_ds = image_dataset_from_directory(
    f"{data_dir_path}/train",
    label_mode="categorical",
    interpolation="bicubic",
    batch_size=batch_size,
    image_size=image_size,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
    verbose=True
)

val_ds = image_dataset_from_directory(
    f"{data_dir_path}/val",
    label_mode="categorical",
    interpolation="bicubic",
    batch_size=batch_size,
    image_size=image_size,
    crop_to_aspect_ratio=True,
    shuffle=False,
    verbose=True
)

test_ds = image_dataset_from_directory(
    f"{data_dir_path}/test",
    label_mode="categorical",
    interpolation="bicubic",
    batch_size=batch_size,
    image_size=image_size,
    crop_to_aspect_ratio=True,
    shuffle=False,
    verbose=True
)


Found 10672 files belonging to 23 classes.
Found 1334 files belonging to 23 classes.
Found 1334 files belonging to 23 classes.


In [8]:
# Params
augmentation_layer = Sequential(
    [
        layers.Rescaling(1./255),
        layers.RandomBrightness(factor=0.1, value_range=(0.0, 1.0)),
        layers.RandomFlip(),
        layers.RandomRotation(factor=0.1, fill_mode="reflect")
    ],
    name="augmentation_layer"
)

# Formato: (filtros, kernel, stride)
# Usamos strides=2 para reduzir a imagem em vez de MaxPool (mais moderno)
conv_setup = [
    (64, (7,7), 2),   # Camada inicial "Stem" (captura specs gerais)
    (64, (3,3), 1),   # Bloco Residual 1
    (128, (3,3), 2),  # Bloco Residual 2 (reduz dimensão)
    (128, (3,3), 1),  # Bloco Residual 3
    (256, (3,3), 2),  # Bloco Residual 4 (reduz dimensão)
    (256, (3,3), 1),  # Bloco Residual 5
    (512, (3,3), 2),  # Bloco Residual 6 (alta abstração)
    (512, (3,3), 1)   # Bloco Residual 7
]

# Camadas densas após o GAP
dense_setup = [1024, 512]
n_classes = 23

In [9]:
# add L2 weight decay to the optimizer directly, don't add a new loss term
model = MyCNN(augmentation_layer=augmentation_layer, conv_configs=conv_setup, dense_configs=dense_setup, num_classes=n_classes)
optimizer = SGD(learning_rate=0.01, name="optimizer", weight_decay=0.01)
loss = CategoricalCrossentropy(name="loss")

In [10]:
# metrics
categorical_accuracy = CategoricalAccuracy(name="accuracy")
auc = AUC(name="auc")
f1_score = F1Score(average="macro", name="f1_score")
metrics = [categorical_accuracy, auc, f1_score]

In [11]:
# traces the computation
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

In [12]:
# callbacks
root_dir_path = Path(".")
checkpoint_file_path = root_dir_path / "checkpoint.keras"
metrics_file_path = root_dir_path = root_dir_path / "metrics.csv"

checkpoint_callback = ModelCheckpoint(
    checkpoint_file_path,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_file_path)

In [13]:
# What is a learning rate scheduler ?
def exp_decay_lr_scheduler(
    epoch: int,
    current_lr: float,
    factor: float = 0.975,
) -> float:
    """
    Exponential decay learning rate scheduler
    """

    current_lr *= factor

    return current_lr

lr_callback = tf.keras.callbacks.LearningRateScheduler(exp_decay_lr_scheduler, verbose=1)

In [14]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_callback
]

In [ ]:
# train the model
_ = model.fit(
    train_ds,
    validation_data=val_ds,
    batch_size=batch_size,
    epochs=epochs,
    class_weight=loaded_weights,
    callbacks=callbacks,
    verbose=2
)


Epoch 1: LearningRateScheduler setting learning rate to 0.009749999782070516.
Epoch 1/64


In [ ]:
# evaluate on the test set
model.evaluate(
    test_ds,
    batch_size=batch_size,
    return_dict=True,
    verbose=0
)

In [ ]:
acc = history.history['categorical_accuracy'] 
val_acc = history.history['val_categorical_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Graph 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='blue')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='orange')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.6)

# Graph 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='blue')
plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()